In [1]:
import json
import pandas as pd
import os

In [2]:
df_lm = pd.read_csv('/home/erik/Riksarkivet/Projects/fastighet/data/LM_2008.txt', sep=';', encoding='latin-1', dtype=str, index_col=[3,4,5,6])

In [3]:
with open('/home/erik/Riksarkivet/Projects/fastighet/output/iteration3_htr_epoch3_with_gt_links.json', 'r') as f:
    htr_dict = json.load(f)

In [4]:
gkomm_dict = {}

for batch in htr_dict.keys():
    gkomms = []
    
    for page in htr_dict[batch].keys():

        if htr_dict[batch][page] is None:
            continue

        for inst in htr_dict[batch][page]:
            #gkomm = inst['gts'][0].split(';')[0]
            gkomms_page = [x[0].split(';')[0] for x in inst['gts']]
            for gk in gkomms_page:
                if gk not in gkomms:
                    gkomms.append(gk)
            
            #if gkomm not in gkomms:
                #gkomms.append(gkomm)

    gkomm_dict[batch] = gkomms

In [5]:
gkomm_dict['10002390']

['O-GÖTEBORG', 'O-VÄSTRA FRÖLUNDA', 'V', 'G']

In [6]:


for i, batch in enumerate(htr_dict.keys()):
    try:
        gkomm_list = gkomm_dict[batch]
    except:
        print('error')

    print(gkomm_list)

    no_of_non_disamb = 0
    no_of_disamb = 0
    no_of_misses = 0
    no_of_ambiguities = 0
    no_of_hits = 0

    for page in htr_dict[batch].keys():

        if htr_dict[batch][page] is not None and htr_dict[batch][page][0]['det_prob'] is not None:

            for inst in htr_dict[batch][page]:

                inst['pred_links'] = []

                try:
                    trakt = inst['pred'].split(';')[0].upper()
                    block = inst['pred'].split(';')[1].upper()
                    enhet = inst['pred'].split(';')[2].upper()

                except:
                    continue
                
                no_of_val_hits = 0
                
                for gk in gkomm_list:  
                    try:
                        df_test  = df_lm.loc[(gk, trakt, block, enhet)]
                    except:
                        continue
                    
                    if len(df_test) > 1:
                        no_of_non_disamb += 1
                    
                    elif len(df_test) == 1:
                        inst['pred_links'].append((gk + ';' + trakt + ';' + block + ';' + enhet, str(df_test['FNR'].iloc[0])))
                        no_of_val_hits += 1
                        no_of_hits += 1

                if len(inst['pred_links']) > 1:
                    no_of_ambiguities += 1

                    

    print(str(i) + ':' + batch + ':' + str(no_of_hits) + ':' + str(no_of_ambiguities))

['G-VÄXJÖ', 'V']
ipykernel_launcher:35: PerformanceWarning: indexing past lexsort depth may impact performance.
0:10001009:506:0
['G-VÄXJÖ', 'V']
1:10001059:597:0
['G-SKATELÖV', 'S']
2:10001239:321:0
['G-ÅSEDA', 'Å']
3:10001419:507:0
['G-ÄLGHULT', 'Ä']
4:10001429:364:0
['Y-NJURUNDA', 'N']
5:10001490:467:0
['Y-NJURUNDA']
6:10001510:487:0
['Y-STÖDE', 'S']
7:10001722:389:0
['Y-LJUSTORP']
8:10001933:316:0
['O-ASKIM', 'A']
9:10002043:266:0
['O-GÖTEBORG', 'O-VÄSTRA FRÖLUNDA', 'V', 'G']
10:10002390:326:0
['O-GÖTEBORG', 'G']
11:10002455:163:0
['U-VÄSTERÅS']
12:10003120:344:0
['O-PARTILLE', 'P']
13:10003246:146:0
['O-PARTILLE', 'P']
14:10003296:34:0
['O-MÖLNDAL', 'M', 'O-FÄSSBERG', 'F']
15:10003466:289:0
['O-UDDEVALLA', 'O-BÄVE', 'U']
16:10003654:847:0
['U', 'O-UDDEVALLA']
17:10003664:393:0
['O-LANE-RYR', 'L']
18:10003724:558:0
['O-LYSEKIL', 'L', 'O-LYSE', 'O-SKAFTÖ', 'O-ASKUM', 'A', 'O-TOSSENE']
19:10003774:308:0
['O-SKAFTÖ', 'S']
20:10003814:277:0
['O-ASKUM']
21:10003833:48:0
['O-ASKUM', 'O-K

In [13]:
df_test['FNR'].iloc[0]

'120230473'

In [7]:
with open('/home/erik/Riksarkivet/Projects/fastighet/output/iteration3_link_epoch3.json', 'w', encoding='utf8') as f:
    j = json.dumps(htr_dict, indent = 4, ensure_ascii=False)
    f.write(str(j))